# NIDS dataset reachability check

Confirms that every dataset the NIDS assignments read is reachable **from this hub** -- including the
six that live on the in-cluster Ceph gateway and therefore cannot be checked from a laptop.

**Run All Cells.** The last line is either `all datasets reachable` or a list of what failed.

This is a reachability pass: a HEAD, the first bytes of an object, a directory listing, a bolt
handshake. It is deliberately *not* a re-implementation of each assignment's
`00-environment-check.ipynb`, which does real reads (a Parquet schema, a Spark session, a Postgres
query) and remains the authoritative check that one assignment is ready to hand out.

Each dataset, and what to do when one fails, is documented in
[datasets/README.md](../datasets/README.md). The laptop-side counterpart is
[scripts/check-datasets.py](../scripts/check-datasets.py).

In [ ]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

In [ ]:
# pelicanfs reads the RouteViews RIB listing over OSDF; neo4j connects to IYP.
# Everything else here is standard library.
#
# On the nids-hub image both are already satisfied and this is a no-op. On the
# hosted NRP hub it really does install into your home directory.
%pip install pelicanfs neo4j

In [ ]:
# --- check runner -------------------------------------------------------------
# Self-contained on purpose: this notebook is handed around on its own, so the
# runner is duplicated here rather than imported from a shared module.
import os
import socket
import sys
import urllib.error
import urllib.request

RESULTS = []  # (label, "ok" | "fail" | "warn")


class check:
    """`with check("label") as c:` -- runs the body, records the outcome, never raises.

    Set `c.note = "..."` inside the body to add detail to the printed line.
    `required=False` downgrades a failure to a warning, which does not block "ready".
    """

    def __init__(self, label, required=True):
        self.label = label
        self.required = required
        self.note = ""

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc is None:
            RESULTS.append((self.label, "ok"))
            print(f"[ ok ] {self.label}" + (f"  ({self.note})" if self.note else ""))
        else:
            RESULTS.append((self.label, "fail" if self.required else "warn"))
            tag = "FAIL" if self.required else "warn"
            print(f"[{tag}] {self.label}  ->  {exc.__class__.__name__}: {exc}")
        return True  # swallow the exception so the remaining checks still run


def http_head(url, timeout=60):
    """HEAD a URL. Raises on a non-2xx status or a connection failure."""
    req = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return response.status, response.headers


def http_first_bytes(url, n=64, timeout=60):
    """Range-GET the first n bytes of a URL -- never pulls the whole object."""
    req = urllib.request.Request(url, headers={"Range": f"bytes=0-{n - 1}"})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return response.read(n)


def human_size(headers):
    size = headers.get("Content-Length")
    return f"{int(size) / 2**20:.0f} MiB" if size else "size unknown"


def mem_limit_gib():
    """This pod's memory limit in GiB (cgroup v2, then v1). None if unlimited."""
    try:
        raw = open("/sys/fs/cgroup/memory.max").read().strip()
        if raw != "max":
            return int(raw) / 2**30
    except OSError:
        pass
    try:
        raw = int(open("/sys/fs/cgroup/memory/memory.limit_in_bytes").read().strip())
        if raw < 2**60:
            return raw / 2**30
    except OSError:
        pass
    return None


def report(subject=""):
    """Print the verdict. Call this last."""
    failed = [label for label, state in RESULTS if state == "fail"]
    warned = [label for label, state in RESULTS if state == "warn"]
    tail = f" for {subject}" if subject else ""
    print()
    for label in warned:
        print(f"warning: {label} did not pass -- not required, see the note above")
    if failed:
        print(f"NOT ready{tail} - {len(failed)} of {len(RESULTS)} checks failed:")
        for label in failed:
            print(f"  - {label}")
        print("\nWhat each failure means: nids-setup/docs/5_verify_hub.md")
    else:
        print(f"JupyterHub is ready{tail}")

In [ ]:
# --- in-cluster Ceph (NRP-internal hostname) -----------------------------------
CEPH = "http://rook-ceph-rgw-nautiluss3.rook"


def _ceph(call, path, *args, **kwargs):
    try:
        return call(f"{CEPH}/{path}", *args, **kwargs)
    except urllib.error.URLError as exc:
        if isinstance(getattr(exc, "reason", None), socket.gaierror):
            raise RuntimeError(
                f"{CEPH} did not resolve. That hostname only exists inside the NRP "
                "cluster -- this will never work from a laptop."
            ) from None
        raise


def ceph_head(path, **kwargs):
    return _ceph(http_head, path, **kwargs)


def ceph_first_bytes(path, n=64, **kwargs):
    return _ceph(http_first_bytes, path, n, **kwargs)

In [ ]:
# --- dataset coordinates -------------------------------------------------------
# Every value below is the one an assignment actually uses today. When one goes
# stale, the matching datasets/<name>/README.md values table needs updating too.

OSDF_RIB_DIR_BGP = "/routeviews/route-views3/bgpdata/2026.05/RIBS"
OSDF_RIB_DIR_TELESCOPE = "/routeviews/route-views3/bgpdata/2026.06/RIBS"
RPKI_URL = "https://ftp.ripe.net/ripe/rpki/afrinic.tal/2023/03/01/roas.csv.xz"
OI_BUCKET_URL = "https://object.openintel.nl/openintel-public"
ANYCAST_URL = "https://manycast.net/api/v1/export/IPv4-latest.parquet"
IYP_URI = "neo4j://iyp-bolt.ihr.live:7687"

AS_CONE = "caida/as-relationships/20260501.ppdc-ases.txt.bz2"
AS2ORG = "caida/as2org/as2org.jsonl"
IRR_OBJECT = "caida/routing/irr_dumps/2023-03-01/ftp.radb.net/radb/dbase/radb.db.gz"
PREFIX2AS_OBJECT = "caida/routing/routeviews-prefix2as/2023/03/routeviews-rv2-20230301-1200.pfx2as.gz"
PCAP_A = "caida/ucsd-nt/sample_062026/ucsd-nt-sub.1782463980.anon.pcap.gz"
PCAP_B = "caida/ucsd-nt/sample_062026/ucsd-nt-sub.1782464400.anon.pcap.gz"
GEOIP_DB = "caida/geolocation/maxmind/2026-06-24.GeoLite2-City.mmdb.gz"

In [ ]:
print("--- in-cluster Ceph: only reachable from inside NRP ---\n")

with check("ceph: customer cone, ppdc-ases (ASN, BGP)") as c:
    head = ceph_first_bytes(AS_CONE, 3)
    assert head == b"BZh", f"expected a bz2 stream, got {head!r}"
    status, headers = ceph_head(AS_CONE)
    c.note = f"BZh ok, {human_size(headers)}"

with check("ceph: as2org.jsonl (ASN, BGP)") as c:
    import json as _json
    first = ceph_first_bytes(AS2ORG, 8192).decode("utf-8", "replace").splitlines()[0]
    _json.loads(first)
    c.note = "first line parses as JSON"

with check("ceph: irr whois dumps (IRR)") as c:
    head = ceph_first_bytes(IRR_OBJECT, 2)
    assert head == b"\x1f\x8b", f"expected gzip, got {head!r}"
    status, headers = ceph_head(IRR_OBJECT)
    c.note = f"gzip ok, radb.db.gz {human_size(headers)}"

with check("ceph: routeviews prefix2as (IRR)") as c:
    head = ceph_first_bytes(PREFIX2AS_OBJECT, 2)
    assert head == b"\x1f\x8b", f"expected gzip, got {head!r}"
    c.note = "gzip ok"

with check("ceph: ucsd-nt pcap samples, both (TELESCOPE)") as c:
    head = ceph_first_bytes(PCAP_A, 2)
    assert head == b"\x1f\x8b", f"expected gzip, got {head!r}"
    status_a, headers_a = ceph_head(PCAP_A)
    status_b, headers_b = ceph_head(PCAP_B)
    c.note = f"{human_size(headers_a)} + {human_size(headers_b)}"

with check("ceph: maxmind geolite2 (TELESCOPE)") as c:
    head = ceph_first_bytes(GEOIP_DB, 2)
    assert head == b"\x1f\x8b", f"expected gzip, got {head!r}"
    status, headers = ceph_head(GEOIP_DB)
    c.note = f"{human_size(headers)} (no MaxMind account needed)"

In [ ]:
print("--- external egress ---\n")

with check("osdf: routeviews rib listings, both months (BGP, TELESCOPE)") as c:
    from pelicanfs.core import OSDFFileSystem
    fs = OSDFFileSystem()
    bgp = fs.ls(OSDF_RIB_DIR_BGP)
    telescope = fs.ls(OSDF_RIB_DIR_TELESCOPE)
    assert bgp, f"{OSDF_RIB_DIR_BGP} listed empty"
    assert telescope, f"{OSDF_RIB_DIR_TELESCOPE} listed empty"
    c.note = f"{len(bgp)} objects (2026.05), {len(telescope)} objects (2026.06)"

with check("ftp.ripe.net: rpki roas (IRR)") as c:
    # The only assignment reaching this host. A namespace that allows CAIDA and
    # OSDF but not RIPE fails here and nowhere else.
    status, headers = http_head(RPKI_URL)
    c.note = f"HTTP {status}, {human_size(headers)}"

with check("object.openintel.nl: bucket is anonymously readable (DNS)") as c:
    # HEAD on the S3 root is 403 by design; the bucket URL is the smallest
    # request proving anonymous access works. The real Parquet read needs Spark
    # and the S3A jars -- that is nids-dns-ecosystem's own check.
    status, _ = http_head(OI_BUCKET_URL)
    c.note = f"HTTP {status}, openintel-public"

with check("manycast.net: anycast census (DNS)") as c:
    # This endpoint answers HEAD with 405 and ignores Range on a GET, so read
    # only the 4-byte Parquet magic rather than pulling ~4 MiB.
    head = http_first_bytes(ANYCAST_URL, 4)
    assert head == b"PAR1", f"expected a Parquet file, got {head!r}"
    c.note = "PAR1 magic ok"

with check("public dns resolution (DNS)") as c:
    # nids-dns-ecosystem resolves real name servers at runtime, which is a
    # separate requirement from reading the OpenINTEL archive.
    socket.getaddrinfo("www.caida.org", 443)

In [ ]:
# Neither of these is part of the NRP hub's job, so both are warnings.
print("--- other environments: informational ---\n")

with check("iyp: public bolt endpoint (IYP)", required=False) as c:
    from neo4j import GraphDatabase
    db = GraphDatabase.driver(IYP_URI, auth=None)
    try:
        db.verify_connectivity()
    finally:
        db.close()
    c.note = IYP_URI

with check("itdk: postgres instance (ITDK)", required=False) as c:
    # Credential-gated, and no instance is deployed yet -- see
    # datasets/itdk-postgres/.
    dsn = os.environ.get("ITDK_READ_DSN")
    if not dsn:
        raise RuntimeError("skipped: no ITDK_READ_DSN set (see datasets/itdk-postgres/)")
    from sqlalchemy import create_engine, text
    engine = create_engine(dsn)
    with engine.connect() as conn:
        tables = conn.execute(
            text("SELECT table_name FROM information_schema.tables "
                 "WHERE table_schema = 'caida_itdk' ORDER BY table_name")
        ).scalars().all()
    assert tables, "schema caida_itdk has no tables"
    c.note = f"{len(tables)} tables"  # the DSN is never printed

print("\nnot checked here: ucsdnt-expanse-flowtuple runs on SDSC Expanse via Slurm,")
print("not on this hub. See datasets/ucsdnt-expanse-flowtuple/.")

In [ ]:
report()